In [30]:
#GPU CHECK


import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA RTX A4000


In [31]:
#LOAD DATASET


import json
import pandas as pd

with open("bert_dataset.json","r",encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(df.head())
print(df.label.value_counts())

                                                text       label
0      bruh send more pics or i leak them. i mean it  sextortion
1              nah send more pics or i leak them 😏 💀  sextortion
2  Do what i say or everyone sees them. dont try ...  sextortion
3  fr im the only one who understands you. you ca...    grooming
4                             Bruh watch your back 💀      threat
label
sextortion    1500
grooming      1500
threat        1500
sexual        1500
bullying      1500
toxicity      1500
hate          1500
coercion      1500
Name: count, dtype: int64


In [32]:
# ENCODE LABELS

labels = sorted(df["label"].unique())

label2id = {
    label: idx
    for idx, label in enumerate(labels)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

df["label_id"] = df["label"].map(label2id)

print(label2id)


{'bullying': 0, 'coercion': 1, 'grooming': 2, 'hate': 3, 'sextortion': 4, 'sexual': 5, 'threat': 6, 'toxicity': 7}


In [33]:
# Stratified Split - Splits the data into train/val/test sets

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label_id"],
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.10,
    stratify=train_df["label_id"],
    random_state=42
)

print(len(train_df))
print(len(val_df))
print(len(test_df))

9180
1020
1800


In [34]:
# Convert to Hugging Face Dataset - Loads the BERTweet tokenizer from HuggingFace.


from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

In [35]:
# Load Tokenizer vinai/bertweet-base - It was trained on Twitter-like text and handles slang much better.


from transformers import AutoTokenizer

MODEL_NAME = "vinai/bertweet-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


In [36]:
# Tokenize

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

In [37]:
# APPLYING TOKENIZATION 

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/9180 [00:00<?, ? examples/s]

Map:   0%|          | 0/1020 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

In [38]:
#Rename Label Column


train_ds = train_ds.rename_column(
    "label_id",
    "labels"
)

val_ds = val_ds.rename_column(
    "label_id",
    "labels"
)

test_ds = test_ds.rename_column(
    "label_id",
    "labels"
)

In [39]:
# Torch Format - Converts dataset columns to PyTorch tensors. 


train_ds.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

val_ds.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

test_ds.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)


In [40]:
#Load BERTweet


from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=9,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.decoder.weight      | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [41]:
# Metrics - Defines accuracy, F1, precision, recall.


import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_fscore_support

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="macro"
        )
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "macro_f1": f1,
        "precision": precision,
        "recall": recall
    }

In [42]:
# Training Arguments   For your RTX 2080Ti / RTX A4000: Sets all training hyperparameters.


from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./guardlink_model",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=5,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    greater_is_better=True,

    logging_steps=50,

    fp16=False
)

In [43]:
# Trainer - Initializes the Trainer object.


from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_ds,
    eval_dataset=val_ds,

    compute_metrics=compute_metrics
)

In [44]:
#  Train


trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision,Recall
1,0.016207,0.009603,1.000000,1.000000,1.000000,1.000000
2,0.005235,0.003200,1.000000,1.000000,1.000000,1.000000
3,0.002796,0.001807,1.000000,1.000000,1.000000,1.000000
4,0.010858,0.001314,1.000000,1.000000,1.000000,1.000000
5,0.001745,0.001164,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2870, training_loss=0.0668253544211803, metrics={'train_runtime': 390.4808, 'train_samples_per_second': 117.547, 'train_steps_per_second': 7.35, 'total_flos': 1509681004646400.0, 'train_loss': 0.0668253544211803, 'epoch': 5.0})

In [ ]:
# Load from Checkpoint (relative path) - not whole model 
 
# from transformers import AutoTokenizer
# from transformers import AutoModelForSequenceClassification

# CHECKPOINT = "./guardlink_model/checkpoint-3230"

# model = AutoModelForSequenceClassification.from_pretrained(
#     CHECKPOINT
# )

# tokenizer = AutoTokenizer.from_pretrained(
#     "vinai/bertweet-base"
# )

# print("Model loaded successfully!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Model loaded successfully!


In [46]:
id2label = model.config.id2label

In [47]:
import torch

id2label = model.config.id2label

text = "watch your back"

# move inputs to the same device as the model
device = next(model.parameters()).device

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=64
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=1).item()

print("Prediction:", id2label[pred])

Prediction: threat


In [49]:
# EVALUATE on Test Set

results = trainer.evaluate(test_ds)

print(results)


from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(test_ds)

preds = np.argmax(
    predictions.predictions,
    axis=1
)

print(
    classification_report(
        predictions.label_ids,
        preds,
        target_names=labels
    )
)

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Precision,Recall
0.001745,0.009600,5,1.000000,1.000000,1.000000,1.000000


{'eval_loss': 0.009600482881069183, 'eval_accuracy': 1.0, 'eval_macro_f1': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0}


              precision    recall  f1-score   support

    bullying       1.00      1.00      1.00       225
    coercion       1.00      1.00      1.00       225
    grooming       1.00      1.00      1.00       225
        hate       1.00      1.00      1.00       225
  sextortion       1.00      1.00      1.00       225
      sexual       1.00      1.00      1.00       225
      threat       1.00      1.00      1.00       225
    toxicity       1.00      1.00      1.00       225

    accuracy                           1.00      1800
   macro avg       1.00      1.00      1.00      1800
weighted avg       1.00      1.00      1.00      1800



In [50]:
print("Total samples:", len(df))
print("Unique texts:", df["text"].nunique())
print("Duplicates:", df.duplicated("text").sum())

Total samples: 12000
Unique texts: 12000
Duplicates: 0


In [ ]:
#Load Model from Absolute Path (checkpoint)

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    r"D:\USAII\guardlink_model\checkpoint-3230"
)

print("Model loaded successfully")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully


In [ ]:
#Load Tokenizer Separately

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "vinai/bertweet-base"
)

print("Tokenizer loaded")

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Tokenizer loaded


In [52]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base")
print("Tokenizer loaded")

device = next(model.parameters()).device

tests = [
    "If you don't stop, something bad will happen.!!!",
    "I want to see your body, dm me.",
    "Your race ruins everything.!!!",
    "You must do this for me because of our secret.!!!",
    "Stop embarrassing yourself, incel."
]

for text in tests:

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=64
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()
    label = model.config.id2label[pred]

    print(f"Text:  {text}")
    print(f"Label: {label}")
    print()

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Tokenizer loaded
Text:  If you don't stop, something bad will happen.!!!
Label: threat

Text:  I want to see your body, dm me.
Label: sexual

Text:  Your race ruins everything.!!!
Label: hate

Text:  You must do this for me because of our secret.!!!
Label: grooming

Text:  Stop embarrassing yourself, incel.
Label: toxicity



In [7]:
#Top-3 Probability Function


import torch

def predict_top3(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=64
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)[0]

    top_probs, top_ids = torch.topk(probs, k=3)

    print(f"\nTEXT: {text}")
    print("-" * 50)

    for prob, idx in zip(top_probs, top_ids):

        label = model.config.id2label[idx.item()]

        print(
            f"{label:<12} {prob.item()*100:.2f}%"
        )
        
        predict_top3("send me your homework")
predict_top3("can you send another photo of the dog")
predict_top3("i miss talking to you")


TEXT: can you send another photo of the dog
--------------------------------------------------
sexual       99.99%

TEXT: send me your homework
--------------------------------------------------
sexual       100.00%

TEXT: send me your homework
--------------------------------------------------
sexual       100.00%

TEXT: send me your homework
--------------------------------------------------
sexual       100.00%

TEXT: send me your homework
--------------------------------------------------
sexual       100.00%

TEXT: send me your homework
--------------------------------------------------
sexual       100.00%

TEXT: send me your homework
--------------------------------------------------
sexual       100.00%

TEXT: send me your homework
--------------------------------------------------
sexual       100.00%

TEXT: send me your homework
--------------------------------------------------
sexual       100.00%

TEXT: send me your homework
-----------------------------------------------

KeyboardInterrupt: 

KeyboardInterrupt: 

In [53]:
#Saves the final model and tokenizer.


FINAL_MODEL_DIR = r"D:\USAII\guardlink_final_model"

model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [ ]:
#RELOAD MODEL 

from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer

test_model = AutoModelForSequenceClassification.from_pretrained(
    r"D:\USAII\guardlink_final_model"
)

test_tokenizer = AutoTokenizer.from_pretrained(
    r"D:\USAII\guardlink_final_model"
)

print("Reload successful")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Reload successful


In [54]:
# Save Label Mapping to JSON

import json

mapping = {
    "id2label": model.config.id2label,
    "label2id": model.config.label2id
}

with open(
    r"D:\USAII\guardlink_final_model\label_mapping.json",
    "w"
) as f:
    json.dump(mapping, f, indent=4)

print("Label mapping saved")

Label mapping saved


In [55]:
import os

for file in os.listdir(r"D:\USAII\guardlink_final_model"):
    print(file)

added_tokens.json
bpe.codes
config.json
label_mapping.json
model.safetensors
tokenizer_config.json
vocab.txt
